# Togather — Exploratory Data Analysis

**Data:** event requests & supplier quotes (2024)  
**Goal:** understand data quality, distributions, and conversion patterns to inform 2025 strategy.

## 0. Setup

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

FIGURES = '../outputs/figures/'
RAW     = '../data/raw/'

In [ ]:
def load_requests(path=RAW + 'food_requests.xlsx'):
    """Row 0 is a junk filename row; actual headers are on row 1."""
    df = pd.read_excel(path, header=1)
    df.columns = ['event_request_id', 'created', 'region', 'priority_tag', 'request_budget']
    df['created'] = pd.to_datetime(df['created'], format='mixed')
    df['priority_tag'] = df['priority_tag'].str.strip()
    return df

def load_quotes(path=RAW + 'food_quotes.xlsx'):
    df = pd.read_excel(path)
    df.columns = [
        'quote_id', 'supplier_id', 'event_request_id',
        'quote_created', 'booked', 'supplier_region',
        'quote_price', 'supplier_primary_tag'
    ]
    df['quote_created'] = pd.to_datetime(df['quote_created'], format='mixed')
    df['booked'] = df['booked'].astype(int)
    df['supplier_primary_tag'] = df['supplier_primary_tag'].str.strip()
    return df

requests = load_requests()
quotes   = load_quotes()
print('Requests:', requests.shape)
print('Quotes:  ', quotes.shape)

---
## 1. Event Requests — standalone

### 1.1 Schema & data quality

In [ ]:
print(requests.dtypes)
print()
print('Null counts:')
print(requests.isnull().sum())
print()
print('Date range:', requests['created'].min(), '→', requests['created'].max())
print('Unique requests:', requests['event_request_id'].nunique())
requests.head()

### 1.2 Monthly request volume

In [ ]:
monthly = requests.set_index('created').resample('ME')['event_request_id'].count()

fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(ax=ax, marker='o')
ax.set_title('Monthly Event Request Volume')
ax.set_xlabel('')
ax.set_ylabel('Number of requests')
plt.tight_layout()
plt.savefig(FIGURES + '01_monthly_requests.png')
plt.show()

### 1.3 Regional distribution

In [ ]:
region_counts = requests['region'].value_counts(dropna=False)
print(f'Missing region: {requests["region"].isnull().sum()} ({requests["region"].isnull().mean():.1%})')

fig, ax = plt.subplots(figsize=(9, 5))
region_counts.plot(kind='barh', ax=ax)
ax.set_title('Event Requests by Region')
ax.set_xlabel('Number of requests')
plt.tight_layout()
plt.savefig(FIGURES + '02_requests_by_region.png')
plt.show()

### 1.4 Priority tag distribution (top 15)

In [ ]:
tag_counts = requests['priority_tag'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 5))
tag_counts.plot(kind='barh', ax=ax)
ax.set_title('Top 15 Customer Priority Tags')
ax.set_xlabel('Number of requests')
plt.tight_layout()
plt.savefig(FIGURES + '03_priority_tags.png')
plt.show()

### 1.5 Budget distribution

In [ ]:
print(requests['request_budget'].describe().round(2))
p99 = requests['request_budget'].quantile(0.99)
outliers = requests[requests['request_budget'] > p99]
print(f'\nOutliers above 99th percentile (£{p99:,.0f}): {len(outliers)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
requests[requests['request_budget'] <= p99]['request_budget'].hist(bins=40, ax=axes[0])
axes[0].set_title('Budget distribution (excl. top 1%)')
axes[0].set_xlabel('Budget (£)')

requests[requests['request_budget'] <= p99].boxplot(column='request_budget', ax=axes[1])
axes[1].set_title('Budget boxplot (excl. top 1%)')
axes[1].set_ylabel('Budget (£)')

plt.tight_layout()
plt.savefig(FIGURES + '04_budget_distribution.png')
plt.show()

---
## 2. Quotes — standalone

### 2.1 Schema & data quality

In [ ]:
print(quotes.dtypes)
print()
print('Null counts:')
print(quotes.isnull().sum())
print()
print('Date range:', quotes['quote_created'].min(), '→', quotes['quote_created'].max())
print('Unique quotes:    ', quotes['quote_id'].nunique())
print('Unique suppliers: ', quotes['supplier_id'].nunique())
quotes.head()

### 2.2 Overall booking rate

In [ ]:
booking_rate = quotes['booked'].mean()
print(f'Overall quote → booking rate: {booking_rate:.2%}')
print(quotes['booked'].value_counts())

### 2.3 Quotes per supplier

In [ ]:
quotes_per_supplier = quotes.groupby('supplier_id')['quote_id'].count()
print(quotes_per_supplier.describe().round(1))

fig, ax = plt.subplots(figsize=(8, 4))
quotes_per_supplier.clip(upper=quotes_per_supplier.quantile(0.99)).hist(bins=40, ax=ax)
ax.set_title('Quotes per Supplier (excl. top 1%)')
ax.set_xlabel('Number of quotes')
ax.set_ylabel('Suppliers')
plt.tight_layout()
plt.savefig(FIGURES + '05_quotes_per_supplier.png')
plt.show()

### 2.4 Supplier regional distribution

In [ ]:
supplier_region = quotes.drop_duplicates('supplier_id')['supplier_region'].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
supplier_region.plot(kind='barh', ax=ax)
ax.set_title('Unique Suppliers by Region')
ax.set_xlabel('Number of suppliers')
plt.tight_layout()
plt.savefig(FIGURES + '06_suppliers_by_region.png')
plt.show()

### 2.5 Supplier primary tag distribution

In [ ]:
supplier_tags = quotes.drop_duplicates('supplier_id')['supplier_primary_tag'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 5))
supplier_tags.plot(kind='barh', ax=ax)
ax.set_title('Top 15 Supplier Primary Tags (unique suppliers)')
ax.set_xlabel('Number of suppliers')
plt.tight_layout()
plt.savefig(FIGURES + '07_supplier_tags.png')
plt.show()

### 2.6 Quote price distribution & booked vs not-booked

In [ ]:
print(quotes['quote_price'].describe().round(2))

p99_price = quotes['quote_price'].quantile(0.99)
fig, ax = plt.subplots(figsize=(10, 4))
for label, grp in quotes[quotes['quote_price'] <= p99_price].groupby('booked'):
    grp['quote_price'].hist(bins=40, alpha=0.6, ax=ax, label='Booked' if label else 'Not booked')
ax.set_title('Quote Price Distribution: Booked vs Not Booked')
ax.set_xlabel('Quote price (£)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES + '08_quote_price_booked.png')
plt.show()

print('\nMedian price — booked vs not booked:')
print(quotes.groupby('booked')['quote_price'].median().rename({0: 'Not booked', 1: 'Booked'}))

---
## 3. Combined Analysis

In [ ]:
# Left join: keep all requests, attach their quotes
combined = requests.merge(quotes, on='event_request_id', how='left')
print('Combined shape:', combined.shape)
combined.head()

### 3.1 Coverage — requests with and without quotes

In [ ]:
has_quote = combined.groupby('event_request_id')['quote_id'].count().rename('quote_count')
requests_with_quotes = (has_quote > 0).mean()
print(f'Requests that received ≥1 quote: {requests_with_quotes:.1%}')
print(f'Requests with ZERO quotes:       {1 - requests_with_quotes:.1%}')

# Unmet demand by region
unmet = combined[combined['quote_id'].isnull()].groupby('region').size().sort_values(ascending=False)
print('\nUnmet requests by region:')
print(unmet)

### 3.2 Customer region vs supplier region (cross-region quoting)

In [ ]:
region_cross = combined.dropna(subset=['region', 'supplier_region']).groupby(
    ['region', 'supplier_region']
).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(region_cross, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Quote Volume: Customer Region vs Supplier Region')
ax.set_xlabel('Supplier Region')
ax.set_ylabel('Customer Region')
plt.tight_layout()
plt.savefig(FIGURES + '09_region_cross.png')
plt.show()

### 3.3 Budget vs quote price delta

In [ ]:
priced = combined.dropna(subset=['request_budget', 'quote_price']).copy()
priced['price_vs_budget'] = priced['quote_price'] - priced['request_budget']
priced['above_budget'] = priced['price_vs_budget'] > 0

print(f'Quotes above customer budget: {priced["above_budget"].mean():.1%}')
print(f'Median delta (quote - budget): £{priced["price_vs_budget"].median():,.0f}')

p1, p99 = priced['price_vs_budget'].quantile([0.01, 0.99])
fig, ax = plt.subplots(figsize=(9, 4))
priced[priced['price_vs_budget'].between(p1, p99)]['price_vs_budget'].hist(bins=40, ax=ax)
ax.axvline(0, color='red', linestyle='--', label='Budget line')
ax.set_title('Quote Price vs Customer Budget (delta)')
ax.set_xlabel('Quote price − Budget (£)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES + '10_price_vs_budget.png')
plt.show()

### 3.4 Priority tag vs supplier primary tag match rate

In [ ]:
tagged = combined.dropna(subset=['priority_tag', 'supplier_primary_tag'])
match_rate = (tagged['priority_tag'] == tagged['supplier_primary_tag']).mean()
print(f'Tag match rate (customer priority == supplier primary): {match_rate:.1%}')

# Booked rate when tags match vs don't match
booked_by_match = tagged.groupby(
    tagged['priority_tag'] == tagged['supplier_primary_tag']
)['booked'].mean().rename({True: 'Tag match', False: 'Tag mismatch'})
print('\nBooking rate by tag match:')
print(booked_by_match)

### 3.5 Conversion funnel

In [ ]:
total_requests   = requests['event_request_id'].nunique()
quoted_requests  = quotes['event_request_id'].nunique()
booked_requests  = quotes[quotes['booked'] == 1]['event_request_id'].nunique()

funnel = pd.Series({
    'Requests': total_requests,
    'Received ≥1 quote': quoted_requests,
    'Resulted in booking': booked_requests,
})

fig, ax = plt.subplots(figsize=(7, 4))
funnel.plot(kind='bar', ax=ax, color=['#4C72B0', '#55A868', '#C44E52'])
ax.set_title('Conversion Funnel: Requests → Quotes → Bookings')
ax.set_ylabel('Unique event requests')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, v in enumerate(funnel):
    ax.text(i, v + total_requests * 0.01, f'{v:,}', ha='center')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES + '11_conversion_funnel.png')
plt.show()

print(f'Quote coverage rate: {quoted_requests/total_requests:.1%}')
print(f'Booking rate (of quoted): {booked_requests/quoted_requests:.1%}')
print(f'Overall conversion (request → booking): {booked_requests/total_requests:.1%}')

### 3.6 Time to first quote

In [ ]:
first_quote = quotes.groupby('event_request_id')['quote_created'].min().rename('first_quote')
response_df = requests.set_index('event_request_id').join(first_quote).dropna(subset=['first_quote'])
response_df['hours_to_first_quote'] = (
    response_df['first_quote'] - response_df['created']
).dt.total_seconds() / 3600

# Remove negative values (data anomaly)
response_df = response_df[response_df['hours_to_first_quote'] >= 0]

print(response_df['hours_to_first_quote'].describe().round(1))

p95 = response_df['hours_to_first_quote'].quantile(0.95)
fig, ax = plt.subplots(figsize=(9, 4))
response_df[response_df['hours_to_first_quote'] <= p95]['hours_to_first_quote'].hist(bins=40, ax=ax)
ax.set_title('Time to First Quote (hours, excl. top 5%)')
ax.set_xlabel('Hours')
plt.tight_layout()
plt.savefig(FIGURES + '12_time_to_first_quote.png')
plt.show()

### 3.7 Regional conversion rates

In [ ]:
# Requests per region
req_by_region = requests.groupby('region')['event_request_id'].nunique().rename('total_requests')

# Bookings per region (customer region)
booked_per_region = (
    combined[combined['booked'] == 1]
    .groupby('region')['event_request_id'].nunique()
    .rename('bookings')
)

regional = pd.concat([req_by_region, booked_per_region], axis=1).dropna()
regional['conversion_rate'] = regional['bookings'] / regional['total_requests']
regional = regional.sort_values('conversion_rate', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
regional['conversion_rate'].plot(kind='barh', ax=ax)
ax.set_title('Request → Booking Conversion Rate by Customer Region')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(FIGURES + '13_regional_conversion.png')
plt.show()

print(regional)

### 3.8 Tag-level conversion rates (top 15 tags by request volume)

In [ ]:
top_tags = requests['priority_tag'].value_counts().head(15).index

req_by_tag = (
    requests[requests['priority_tag'].isin(top_tags)]
    .groupby('priority_tag')['event_request_id'].nunique()
    .rename('total_requests')
)

booked_by_tag = (
    combined[
        (combined['booked'] == 1) &
        (combined['priority_tag'].isin(top_tags))
    ]
    .groupby('priority_tag')['event_request_id'].nunique()
    .rename('bookings')
)

tag_conv = pd.concat([req_by_tag, booked_by_tag], axis=1).fillna(0)
tag_conv['conversion_rate'] = tag_conv['bookings'] / tag_conv['total_requests']
tag_conv = tag_conv.sort_values('conversion_rate', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
tag_conv['conversion_rate'].plot(kind='barh', ax=ax)
ax.set_title('Request → Booking Conversion Rate by Priority Tag (top 15 by volume)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(FIGURES + '14_tag_conversion.png')
plt.show()

print(tag_conv)

---
## Summary of Data Quality Issues

| Issue | Table | Impact |
|-------|-------|--------|
| Junk header row (file name in row 0) | Requests | Must load with `header=1` |
| Null `region` values | Requests | Affects regional analysis; flag in slides |
| Encoding error on `Crêpes` tag | Quotes | Tag matching may undercount; normalise on load |
| Negative time-to-first-quote values | Combined | Quote timestamp precedes request timestamp — data anomaly |